# 02 — Case Representation
**Tujuan:** Ekstrak metadata & fitur dari setiap putusan, simpan ke `data/processed/`.

```
projek-cbr-hukum/
└── data/
    ├── raw/          ← input  (dari notebook 01)
    └── processed/    ← output (cases.csv & cases.json)
```

In [1]:
import re, sys, json
from pathlib import Path
import pandas as pd
from tqdm import tqdm

BASE          = Path('..').resolve()
RAW_DIR       = BASE / 'data' / 'raw' / 'cleaned'
PROCESSED_DIR = BASE / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_files = sorted(RAW_DIR.glob('case_*.txt'))
print(f'File ditemukan: {len(raw_files)}')

File ditemukan: 60


In [2]:
# ════════════════════════════════════════════════════════════
# FUNGSI EKSTRAKSI METADATA
# ════════════════════════════════════════════════════════════

BULAN = {
    'januari':'01','februari':'02','maret':'03','april':'04',
    'mei':'05','juni':'06','juli':'07','agustus':'08',
    'september':'09','oktober':'10','november':'11','desember':'12'
}

def get_nomor_perkara(t):
    for pat in [
        r'\d{1,4}/pid\.?sus[\w./\-]*',
        r'\d{1,4}/pid[\w./\-]*',
        r'\d{1,4}/pdt[\w./\-]*',
        r'nomor\s*[:\-]?\s*([\d/a-z\.\-\s]{8,60})',
    ]:
        m = re.search(pat, t[:3000], re.IGNORECASE)
        if m:
            val = m.group(0) if m.lastindex is None else m.group(1)
            return re.sub(r'\s+', ' ', val.strip()).upper()[:80]
    return 'N/A'

def get_tanggal(t):
    m = re.search(
        r'(\d{1,2})\s+(januari|februari|maret|april|mei|juni|juli|agustus|'
        r'september|oktober|november|desember)\s+(20\d{2}|19\d{2})',
        t[:5000], re.IGNORECASE
    )
    if m:
        return f"{m.group(3)}-{BULAN.get(m.group(2).lower(),'00')}-{m.group(1).zfill(2)}"
    return 'N/A'

def get_jenis_perkara(t):
    if any(k in t for k in ['narkotika','psikotropika','sabu','ganja','kokain']):
        return 'Pidana Khusus Narkotika'
    if any(k in t for k in ['korupsi','gratifikasi','suap']):
        return 'Pidana Khusus Korupsi'
    if 'wanprestasi' in t or 'perbuatan melawan hukum' in t:
        return 'Perdata'
    if 'pid.sus' in t: return 'Pidana Khusus'
    if '/pid' in t:    return 'Pidana Umum'
    return 'Tidak Terdeteksi'

def get_pasal(t):
    pats = [
        r'pasal\s+\d+\s*(?:ayat\s*\(\d+\)\s*)?(?:jo\.?\s*pasal\s+\d+[^;\n]{0,40})?'
        r'(?:uu\s+(?:nomor|no\.?)\s+\d+\s+tahun\s+\d+[^;\n]{0,40})?',
        r'pasal\s+\d+\s+kuhp(?:er)?',
    ]
    found = []
    for p in pats:
        found += re.findall(p, t, re.IGNORECASE)
    unique = list(dict.fromkeys(re.sub(r'\s+', ' ', x.strip()) for x in found))
    return '; '.join(unique[:5]) if unique else 'N/A'

def get_pihak(t):
    td, jp = 'N/A', 'N/A'
    m1 = re.search(r'(?:terdakwa)\s*[:\-]?\s*([A-Z][A-Za-z\s\.]{3,50})(?:[,;\n]|bin|binti|berumur)', t[:5000])
    if m1: td = m1.group(1).strip()
    m2 = re.search(r'(?:penuntut\s+umum|jaksa)\s*[:\-]?\s*([A-Z][A-Za-z\s\.]{3,50})(?:[,;\n])', t[:5000])
    if m2: jp = m2.group(1).strip()
    return td, jp

def get_vonis(t):
    pats = [
        r'pidana\s+penjara\s+selama\s+\d+[^.;]{0,60}',
        r'\d+\s*(?:tahun|bulan)\s*(?:penjara|kurungan)',
        r'denda\s+rp[^.;]{0,60}',
        r'(?:bebas|tidak\s+terbukti|lepas\s+dari\s+tuntutan)',
    ]
    found = []
    for p in pats:
        found += re.findall(p, t, re.IGNORECASE)[:2]
    return '; '.join(found[:3]) if found else 'N/A'

def get_amar(t):
    m = re.search(
        r'(?:mengadili|m\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i|amar\s+putusan)'
        r'[\s:]*(.{100,1200}?)(?:ditetapkan|demikian|[A-Z]{5,}\s*:|\Z)',
        t, re.DOTALL | re.IGNORECASE
    )
    return re.sub(r'\s+', ' ', m.group(1)).strip()[:700] if m else 'N/A'

def get_fakta(t):
    m = re.search(
        r'(?:dakwaan|tentang\s+duduknya\s+perkara|menimbang)'
        r'[\s:]*(.{200,1200}?)(?:menimbang|tuntutan|\n\n\n|\Z)',
        t, re.DOTALL | re.IGNORECASE
    )
    if m: return re.sub(r'\s+', ' ', m.group(1)).strip()[:600]
    return re.sub(r'\s+', ' ', t[len(t)//4 : len(t)//4 + 600])

def assign_label(vonis):
    v = vonis.lower()
    if any(k in v for k in ['bebas','tidak terbukti','lepas']): return 'bebas'
    m = re.search(r'(\d+)\s*tahun', v)
    if m:
        y = int(m.group(1))
        return 'ringan' if y <= 4 else ('sedang' if y <= 10 else 'berat')
    return 'tidak_diketahui'

print('Fungsi ekstraksi siap ')

Fungsi ekstraksi siap 


In [3]:
# ════════════════════════════════════════════════════════════
# PROSES SEMUA KASUS
# ════════════════════════════════════════════════════════════
records = []

for fpath in tqdm(raw_files, desc='Ekstraksi metadata'):
    text = fpath.read_text(encoding='utf-8')
    td, jp = get_pihak(text)

    records.append({
        'case_id'        : fpath.stem,
        'no_perkara'     : get_nomor_perkara(text),
        'tanggal'        : get_tanggal(text),
        'jenis_perkara'  : get_jenis_perkara(text),
        'terdakwa'       : td,
        'penuntut_umum'  : jp,
        'pasal'          : get_pasal(text),
        'vonis'          : get_vonis(text),
        'amar_putusan'   : get_amar(text),
        'ringkasan_fakta': get_fakta(text),
        'word_count'     : len(text.split()),
        'text_full'      : text,
    })

df = pd.DataFrame(records)
df['label_vonis']   = df['vonis'].apply(assign_label)
df['text_combined'] = (
    df['ringkasan_fakta'].fillna('') + ' ' +
    df['pasal'].fillna('')            + ' ' +
    df['amar_putusan'].fillna('')
)

print(f'Total kasus: {len(df)}')
print('Distribusi label:')
print(df['label_vonis'].value_counts().to_string())

Ekstraksi metadata:   0%|          | 0/60 [00:00<?, ?it/s]

Ekstraksi metadata: 100%|██████████| 60/60 [00:00<00:00, 68.67it/s]

Total kasus: 60
Distribusi label:
label_vonis
tidak_diketahui    45
bebas              15


In [4]:
# ════════════════════════════════════════════════════════════
# SIMPAN OUTPUT
# ════════════════════════════════════════════════════════════
cols_csv = [c for c in df.columns if c != 'text_full']

csv_path  = PROCESSED_DIR / 'cases.csv'
json_path = PROCESSED_DIR / 'cases.json'

df[cols_csv].to_csv(csv_path, index=False, encoding='utf-8')
df.to_json(json_path, orient='records', force_ascii=False, indent=2)

print(f' Tersimpan:')
print(f'   {csv_path}')
print(f'   {json_path}')



 Tersimpan:
   D:\Kuliah\Tugas Penalaran Komputer\Penalaran-Komputer_SubCpmk3\data\processed\cases.csv
   D:\Kuliah\Tugas Penalaran Komputer\Penalaran-Komputer_SubCpmk3\data\processed\cases.json


In [5]:
preview_cols = ['case_id','no_perkara','tanggal','jenis_perkara','pasal','vonis']
df[preview_cols].head(60)

,case_id,no_perkara,tanggal,jenis_perkara,pasal,vonis
0,case_001,1006/PID.B/2024/PN,1974-06-05,Pidana Umum,pasal 170 ayat (1); pasal 170ayat (1); pasal 3...,N/A
1,case_002,1055/PID.B/2024/PN,2005-12-03,Pidana Umum,pasal 355 ayat (2); pasal 351 ayat (3); pasal ...,pidana penjara selama 10 (sepuluh)tahun dan 6 ...
2,case_003,1063/PID.B/2024/PN,1975-08-17,Pidana Umum,pasal 170 ayat (1); pasal 351 ayat (1) jo pasa...,N/A
3,case_004,1078/PID.B/2025/PN,1990-04-29,Pidana Umum,pasal 170 ayat (1); pasal 351 ayat (1); pasal ...,pidana penjara selama 3(tiga) tahun dikurangi ...
4,case_005,1080/PID.B/2024/PN,2006-03-18,Pidana Umum,pasal 170 ayat (1); pasal 351 ayat (1) jo. pas...,pidana penjara selama 2 (dua) tahun dikurangis...
5,case_006,1101/PID.B/2024/PN,1990-12-02,Pidana Umum,pasal 351 ayat (1); pasal 22 ayat (4); pasal 3...,pidana penjara selama 1 (satu) tahun dan 6 (en...
6,case_007,1102/PID.B/2024/PN,2002-04-04,Pidana Umum,pasal 351 ayat (1),pidana penjara selama 2 (dua) tahun dikurangi ...
7,case_008,1114/PID.B/2024/PN,1996-10-25,Pidana Umum,pasal 55 ayat (1); pasal 170 ayat (1); pasal 3...,pidana penjara selama 1 (satu) tahun dikurangi...
8,case_009,1115/PID.B/2024/PN,1987-11-21,Pidana Umum,pasal 55 ayat (1); pasal 170 ayat (1); pasal 3...,pidana penjara selama 1 (satu) tahun dikurangi...
9,case_010,1116/PID.B/2024/PN,1983-10-14,Pidana Umum,pasal 55 ayat (1); pasal 170 ayat (1); pasal 3...,pidana penjara selama 1 (satu) tahun dikurangi...
